In [0]:
spark.sql(
    "DROP TABLE IF EXISTS workspace.gold.dim_magnitude"
)

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_magnitude (
    magnitude_key BIGINT,
    magnitude_type STRING,
    magnitude_category STRING
)
""")

In [0]:
from pyspark.sql import functions as F

magnitude = (
    spark.table(
        "workspace.silver.usgs_earthquakes"
    )
    .select(
        "magnitude_type",
        "magnitude"
    )
    .withColumn(
        "magnitude_category",
        F.when(F.col("magnitude") < 2, "Menor a 2")
         .when(F.col("magnitude") < 4, "2 a menor de 4")
         .when(F.col("magnitude") < 5, "4 a menor de 5")
         .when(F.col("magnitude") < 6, "5 a menor de 6")
         .when(F.col("magnitude") < 7, "6 a menor de 7")
         .otherwise("7 o más")
    )
    .select(
        "magnitude_type",
        "magnitude_category"
    )
    .dropDuplicates()
    .withColumn(
        "magnitude_key",
        F.xxhash64(
            "magnitude_type",
            "magnitude_category"
        )
    )
)

magnitude = magnitude.select(
    "magnitude_key",
    "magnitude_type",
    "magnitude_category"
)

In [0]:
magnitude.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.gold.dim_magnitude")